# Training
The following notebook consists of:
- Training the n-gram model with the different training sets
- Uses nltk library for the n-gram
- Finding the bext model and vocab from the training sets
- Finding the best 'n' number in the n-gram (3,5,7)




In [ ]:
from collections import Counter
from nltk.lm import Lidstone
from nltk.lm import KneserNeyInterpolated
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import Vocabulary
import pickle

'''Import libraries and function to load in the training sets'''
def load_methods(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [line.strip().split() for line in f if line.strip()]
    


In [ ]:
''' Function to build the nltk vocab
unk cutoff set to 3, forcing the vocab to see a token 3 times before being known
if less will be <UNK> '''
def build_nltk_vocab(train_methods, min_freq=3):
    counter = Counter()
    for m in train_methods:
        counter.update(m)
    return Vocabulary(counter, unk_cutoff=min_freq)


In [ ]:
'''Apply the vocab, mapping to <UNK> if not found in vocab'''
def apply_vocab(methods, vocab):
    return [[token if token in vocab else "<UNK>" for token in m] for m in methods]


In [ ]:
'''Train model function
use alpha value of 0.01 to handle the sparsity of 
the Java vocabulary without over-smoothing 
the high-probability transitions between 
common language keywords.'''
def train_model(train_methods, vocab, n, alpha=0.01):
    # Map training data to vocab
    train_methods = apply_vocab(train_methods, vocab)
    
    train_data, padded_sents = padded_everygram_pipeline(n, train_methods)
    
    model = Lidstone(alpha, order=n)
    model.fit(train_data, padded_sents)
    
    return model



In [ ]:
from nltk.util import ngrams

"""Calculates the perplexity of the N-gram model on a given set of Java methods.
model: The trained N-gram model (e.g., Lidstone).
methods: A list of tokenized Java methods.
vocab: The vocabulary built during the training phase.
returns: The calculated perplexity score for the entire input dataset."""

def perplexity(model, methods, vocab):
    # Vocabulary Mapping ensuring all tokens in the test set exist in our vocab.
    # Tokens unseen during training are mapped to the <UNK> token.
    methods = apply_vocab(methods, vocab)
    n = model.order
    
    all_ngrams = []
    for m in methods:
        padded = list(ngrams(
            m,
            n,
            pad_left=True,
            pad_right=True,
            left_pad_symbol="<s>", # Start of sentence/method symbol
            right_pad_symbol="</s>" # End of sentence/method symbol
        ))
        all_ngrams.extend(padded)
    
    return model.perplexity(all_ngrams)



In [ ]:
''' The training pipeline for each training set,
loads in each training set and builds own vocabulary for each
training set.
Test 3,5 and 7 gram for each training set'''
val_methods = load_methods('./dataset/ngram_dataset/validation.txt')

datasets = [
    ("T1", "./dataset/ngram_dataset/train_15k.txt"),
    ("T2", "./dataset/ngram_dataset/train_25k.txt"),
    ("T3", "./dataset/ngram_dataset/train_35k.txt")
]

best_model = None
best_vocab = None
best_info = {"perplexity": float("inf")}

# iterate through each training set in datasets list 
for name, path in datasets:
    print(f"\n=== {name} ===")
    
    train_methods = load_methods(path)
    vocab = build_nltk_vocab(train_methods)
    
    for n in [3, 5, 7]:
        print(f"Training {n}-gram...", end=" ")
        
        # train 
        model = train_model(train_methods, vocab, n)
        # get perplexity
        perp = perplexity(model, val_methods, vocab)
        
        print(f"Val Perplexity: {perp:.4f}")
        
        # Check perplexity scores to current bests to see if new best model was found
        if perp < best_info["perplexity"]:
            best_info = {
                "dataset": name,
                "n": n,
                "perplexity": perp
            }
            best_model = model
            best_vocab = vocab



=== T1 ===
Training 3-gram... Val Perplexity: 40.7931
Training 5-gram... Val Perplexity: 223.7790
Training 7-gram... Val Perplexity: 827.8859

=== T2 ===
Training 3-gram... Val Perplexity: 48.6296
Training 5-gram... Val Perplexity: 282.6958
Training 7-gram... Val Perplexity: 1041.9770

=== T3 ===
Training 3-gram... Val Perplexity: 54.9678
Training 5-gram... Val Perplexity: 333.7070
Training 7-gram... Val Perplexity: 1231.9175


In [ ]:
# Dump the vocab to a file to be used in the test_script.py 
print(f'Saving best model: {best_info["dataset"]}')
with open('best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)


print(f'Saving best vocab from model: {best_info["dataset"]}')
with open('best_vocab.pkl', 'wb') as f:
    pickle.dump(best_vocab, f)

print("Data dumped successfully!")

Saving best model: T1
Saving best vocab from model: T1
Data dumped successfully!
